# Neural Network Pruning

## Learning Objectives
1. Understand weight magnitude pruning and how sparsity is measured
2. Apply unstructured and structured pruning using `torch.nn.utils.prune`
3. Implement iterative magnitude pruning (lottery ticket hypothesis)
4. Analyse accuracy-vs-sparsity trade-offs across different pruning ratios


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import copy
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — Weight Magnitude Pruning from Scratch

In [ ]:
# ── Level 1: magnitude pruning (numpy, no torch) ──────────────────────────
import numpy as np

np.random.seed(42)
W = np.random.randn(4, 8)          # simulated weight matrix
print("Original W (first row):", W[0])

def magnitude_prune(W: np.ndarray, sparsity: float) -> np.ndarray:
    """Zero out the bottom `sparsity` fraction of weights by absolute value."""
    flat = np.abs(W).flatten()
    threshold = np.percentile(flat, sparsity * 100)
    mask = np.abs(W) > threshold
    return W * mask                # pruned weights

def measure_sparsity(W: np.ndarray) -> float:
    """Return fraction of zero entries."""
    return float((W == 0).sum()) / W.size

sparsities = [0.0, 0.3, 0.5, 0.7, 0.9]
for s in sparsities:
    W_pruned = magnitude_prune(W, s)
    actual = measure_sparsity(W_pruned)
    print(f"Target sparsity={s:.1f}  Actual sparsity={actual:.3f}  "
          f"Non-zeros={int((W_pruned != 0).sum())}/{W.size}")

# Visualise magnitude distribution
flat_abs = np.abs(W).flatten()
print(f"\nWeight |w| stats — min:{flat_abs.min():.3f}  "
      f"mean:{flat_abs.mean():.3f}  max:{flat_abs.max():.3f}")


## Level 2 — PyTorch Structured & Unstructured Pruning

In [ ]:
# ── Level 2: torch.nn.utils.prune ─────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import copy

# Tiny MLP for demonstration
class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

model = SmallMLP().to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

def sparsity_of(tensor):
    return float((tensor == 0).sum().item()) / tensor.numel()

print(f"Parameters before pruning: {count_params(model):,}")

# ── Unstructured L1 pruning (weight-level) ───────────────────────
model_unstr = copy.deepcopy(model)
prune.l1_unstructured(model_unstr.fc1, name="weight", amount=0.4)
prune.l1_unstructured(model_unstr.fc2, name="weight", amount=0.4)
prune.l1_unstructured(model_unstr.fc3, name="weight", amount=0.4)

sp1 = sparsity_of(model_unstr.fc1.weight)
sp2 = sparsity_of(model_unstr.fc2.weight)
print(f"Unstructured pruning (40%) — fc1 sparsity: {sp1:.2f}, fc2 sparsity: {sp2:.2f}")

# Remove pruning reparameterization (make permanent)
for module in [model_unstr.fc1, model_unstr.fc2, model_unstr.fc3]:
    prune.remove(module, "weight")

# ── Structured Ln pruning (row/filter-level) ─────────────────────
model_str = copy.deepcopy(model)
prune.ln_structured(model_str.fc1, name="weight", amount=0.3, n=2, dim=0)
print(f"Structured pruning (30%) — fc1 zero rows: "
      f"{int((model_str.fc1.weight_mask.sum(dim=1) == 0).sum())} / "
      f"{model_str.fc1.weight_mask.shape[0]}")

# ── Quick accuracy proxy (random data) ───────────────────────────
x_test = torch.randn(256, 64).to(device)
y_test = torch.randint(0, 10, (256,)).to(device)

def accuracy(m, x, y):
    m.eval()
    with torch.no_grad():
        preds = m(x).argmax(dim=1)
    return (preds == y).float().mean().item()

acc_orig  = accuracy(model,      x_test, y_test)
acc_unstr = accuracy(model_unstr, x_test, y_test)

print(f"\nRandom-init accuracy (baseline)   : {acc_orig:.3f}")
print(f"After 40% unstructured pruning     : {acc_unstr:.3f}")
print("(Accuracy differences are noise on random init — retrain to see real effect)")


## Real-World Example 1 — Iterative Magnitude Pruning (Lottery Ticket)

In [ ]:
# ── RW1: Iterative magnitude pruning — lottery ticket hypothesis ──────────
import torch, copy
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, TensorDataset

# Synthetic classification task
torch.manual_seed(42)
X = torch.randn(1000, 32)
y = (X[:, 0] + X[:, 1] > 0).long()
train_ds = TensorDataset(X[:800], y[:800])
val_ds   = TensorDataset(X[800:], y[800:])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(32, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)
    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

def train_one_epoch(model, loader, opt):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        opt.step()

def eval_acc(model, ds):
    model.eval()
    x, y = ds.tensors
    x, y = x.to(device), y.to(device)
    with torch.no_grad():
        return (model(x).argmax(1) == y).float().mean().item()

# Iterative pruning: prune 20% per round × 5 rounds
ROUNDS      = 5
PRUNE_RATE  = 0.20          # fraction of *remaining* weights per round
TRAIN_EPOCHS = 10

results = []
net = Net().to(device)
orig_state = copy.deepcopy(net.state_dict())  # "winning ticket" initial weights

for rnd in range(ROUNDS):
    # Reset to original weights (lottery ticket rewind)
    net.load_state_dict(copy.deepcopy(orig_state))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(TRAIN_EPOCHS):
        train_one_epoch(net, train_loader, opt)

    acc_before = eval_acc(net, val_ds)

    # Prune
    for module in [net.fc1, net.fc2, net.fc3]:
        prune.l1_unstructured(module, name="weight", amount=PRUNE_RATE)
        prune.remove(module, "weight")

    total_params = sum(p.numel() for p in net.parameters())
    zero_params  = sum((p == 0).sum().item() for p in net.parameters())
    sparsity     = zero_params / total_params

    results.append({"round": rnd + 1, "sparsity": sparsity, "val_acc": acc_before})
    print(f"Round {rnd+1}: sparsity={sparsity:.2%}  val_acc={acc_before:.3f}")

print("\nLottery-ticket summary (acc should stay reasonable even at high sparsity)")


## Real-World Example 2 — BERT Attention-Head Pruning

In [ ]:
# ── RW2: BERT attention head importance scoring ──────────────────────────
# Uses only torch + numpy (no internet download needed)
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

# Simulate a single BERT-style self-attention layer (small dims for speed)
BATCH, SEQ, HIDDEN, N_HEADS = 4, 16, 64, 4
HEAD_DIM = HIDDEN // N_HEADS

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden, n_heads):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = hidden // n_heads
        self.Wq = nn.Linear(hidden, hidden)
        self.Wk = nn.Linear(hidden, hidden)
        self.Wv = nn.Linear(hidden, hidden)
        self.Wo = nn.Linear(hidden, hidden)

    def forward(self, x):
        B, T, H = x.shape
        q = self.Wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.Wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.Wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn   = scores.softmax(dim=-1)
        out    = (attn @ v).transpose(1, 2).reshape(B, T, H)
        return self.Wo(out), attn          # return attn weights for scoring

attn_layer = MultiHeadSelfAttention(HIDDEN, N_HEADS).to(device)

# Compute head importance = mean entropy of attention weights (lower = more focused)
x_in = torch.randn(BATCH, SEQ, HIDDEN).to(device)
_, attn_weights = attn_layer(x_in)       # (B, H, T, T)

def head_entropy(attn_w):
    # Compute mean entropy per head (lower = head is more important/focused).
    eps = 1e-9
    ent = -(attn_w * (attn_w + eps).log()).sum(dim=-1).mean(dim=(0, 2))  # (n_heads,)
    return ent.detach().cpu().numpy()

entropies = head_entropy(attn_weights)
print("Per-head entropy (lower = more focused):")
for h, e in enumerate(entropies):
    print(f"  Head {h}: {e:.4f}")

# Rank heads: prune the highest-entropy (least focused) head
prune_idx = int(np.argmax(entropies))
print(f"\nPruning head {prune_idx} (entropy={entropies[prune_idx]:.4f})")

# Mask out head by zeroing corresponding Wv rows
with torch.no_grad():
    start = prune_idx * HEAD_DIM
    end   = start + HEAD_DIM
    attn_layer.Wv.weight[start:end, :] = 0
    attn_layer.Wv.bias[start:end] = 0

_, attn_after = attn_layer(x_in)
entropies_after = head_entropy(attn_after)
print("\nEntropies after pruning head", prune_idx)
for h, e in enumerate(entropies_after):
    print(f"  Head {h}: {e:.4f}")
print("(Pruned head entropy becomes undefined; others unchanged)")


## Real-World Example 3 — Pruning Ratio vs Accuracy/Latency Trade-off

In [ ]:
# ── RW3: sparsity sweep — accuracy vs latency trade-off ─────────────────
import torch, time, copy
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# Synthetic data
X = torch.randn(800, 64)
y = (X[:, :4].sum(dim=1) > 0).long()
train_loader = DataLoader(TensorDataset(X[:600], y[:600]), batch_size=64, shuffle=True)
X_val, y_val = X[600:].to(device), y[600:].to(device)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x): return self.layers(x)

# Train base model
base = MLP().to(device)
opt = torch.optim.Adam(base.parameters(), lr=1e-3)
for _ in range(20):
    base.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        F.cross_entropy(base(xb), yb).backward()
        opt.step()

def eval_acc_latency(model, x, y, repeats=50):
    model.eval()
    with torch.no_grad():
        acc = (model(x).argmax(1) == y).float().mean().item()
        t0 = time.perf_counter()
        for _ in range(repeats):
            _ = model(x)
        latency_ms = (time.perf_counter() - t0) / repeats * 1000
    return acc, latency_ms

sweep_sparsities = [0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9]
print(f"{'Sparsity':>10}  {'Val Acc':>8}  {'Latency (ms)':>14}")
print("-" * 38)

for sp in sweep_sparsities:
    m = copy.deepcopy(base)
    if sp > 0:
        for layer in m.layers:
            if isinstance(layer, nn.Linear):
                prune.l1_unstructured(layer, name="weight", amount=sp)
                prune.remove(layer, "weight")
    acc, lat = eval_acc_latency(m, X_val, y_val)
    print(f"{sp:>10.0%}  {acc:>8.3f}  {lat:>14.3f}")

print("\nNote: unstructured pruning does not reduce latency on dense hardware;")
print("structured pruning (filter/row removal) is needed for real speedup.")
